In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import re

In [ ]:
# 1.load data and notice the label
# 2.Data augmentation: horizontal flip, Gaussian noise
X, y = [], []
data_dir = "../preprocessing/preprocessed_vols"

for file in os.listdir(data_dir):
    if file.endswith(".npy"):
        match = re.search(r'label([01])', file)
        if not match:
            continue
        label = int(match.group(1))
        vol = np.load(os.path.join(data_dir, file))

        X.append(vol)
        y.append(label)

        X.append(np.flip(vol, axis=3))
        y.append(label)

        noisy = vol + np.random.normal(0, 0.01, vol.shape).astype(np.float32)
        X.append(noisy)
        y.append(label)

In [ ]:
# Normalization and split into training and test sets
X = np.array(X)
y = np.array(y)
X = (X - np.mean(X)) / np.std(X)

# train:80%, test:20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Convert NumPy data to PyTorch tensors
# Convert the feature matrices in the training and test sets from NumPy to PyTorch tensors of 32-bit floating point numbers (float32), which is a common data type for CNN input.The labels are also converted to tensors, specifying the data type as long (int64), which is the type required by CrossEntropyLoss().
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Design a sampler for class imbalance, count the number of samples for each class, calculate and assign weights, and create a weighted sampler
class_sample_count = np.array([np.sum(y_train == t) for t in np.unique(y_train)])
weights = 1. / class_sample_count
sample_weights = weights[y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


In [ ]:
# 5-layer 3D CNN model structure, constructing a 5-layer 3D convolutional network with BatchNorm and Dropout for volumetric MRI data classification.
class Deeper3DCNN(nn.Module):
    def __init__(self, dropout=0.5):
        super(Deeper3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 4, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(4)
        self.conv2 = nn.Conv3d(4, 8, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(8)
        self.conv3 = nn.Conv3d(8, 16, 3, padding=1)
        self.bn3 = nn.BatchNorm3d(16)
        self.conv4 = nn.Conv3d(16, 32, 3, padding=1)
        self.bn4 = nn.BatchNorm3d(32)
        self.conv5 = nn.Conv3d(32, 64, 3, padding=1)
        self.bn5 = nn.BatchNorm3d(64)

        self.pool = nn.MaxPool3d(2)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = None
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = self.pool(torch.relu(self.bn4(self.conv4(x))))
        x = self.pool(torch.relu(self.bn5(self.conv5(x))))

        x = x.view(x.size(0), -1)

        if self.fc1 is None:
            self.fc1 = nn.Linear(x.size(1), 32).to(x.device)

        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

In [ ]:
# Define training and evaluation functions
# lr: learning rate;batch_size; number of samples in each mini-batch; dropout_rate: dropout rate, used to prevent overfitting; epochs: number of training rounds
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_evaluate(lr, batch_size, dropout_rate, epochs):
    model = Deeper3DCNN(dropout=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Create a DataLoader using dynamic batch size, in hyperparameter tuning
    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        sampler=sampler
    )
    test_loader = DataLoader(
        TensorDataset(X_test_tensor, y_test_tensor),
        batch_size=batch_size)

    # Initialize storage list
    train_loss = []
    train_acc = []
    test_acc = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train, total_train = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_loss.append(running_loss / len(train_loader))
        train_acc.append(correct_train / total_train)

        model.eval()
        correct_test, total_test = 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                correct_test += (predicted.cpu() == labels).sum().item()
                total_test += labels.size(0)
        acc = correct_test / total_test
        test_acc.append(acc)

    return train_loss, train_acc, test_acc

In [ ]:
# define the hyperparameter, draw the plot of train, test, loss. create a confusion matrix
# hyperparameter tuning preparation
larning_rate_list = [3e-4]
batch_list = [8]
dropout_list = [0.5]
epochs = 7

for lr in larning_rate_list:
    for batch in batch_list:
        for dropout in dropout_list:
            train_loss, train_acc, test_acc = train_and_evaluate(lr, batch, dropout, epochs)

            plt.plot(range(1, epochs+1), train_loss, label='Training Loss')
            plt.plot(range(1, epochs+1), train_acc, label='Training Accuracy')
            plt.plot(range(1, epochs+1), test_acc, label='Test Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Value')
            plt.title(f'Training Loss, Train Accuracy, Test Accuracy per Epoch\n(lr={lr}, batch={batch}, dropout={dropout})')
            plt.legend()
            plt.show()

            final_acc = test_acc[-1]
            print(f"lr={lr}, batch={batch}, dropout={dropout} -> acc={final_acc:.2%}")
